In [1]:
import random
import numpy as np
import os

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed(42)


In [2]:
#pip install sentence-transformers pandas numpy scikit-learn torch

# Import Libraries

In [3]:
import pandas as pd
import numpy as np
import re
import torch
import pickle
from sentence_transformers import SentenceTransformer, util

# Load Dataset

In [4]:
df = pd.read_csv("input/amazon-eco-friendly-products-dataset/amazon_eco-friendly_products.csv")

df.head()

,id,title,name,category,material,brand,price,rating,reviewsCount,description,url,img_url,inStock,inStockText
0,B0CWH366KJ,"Agfabric Natural Jute Erosion Control, 16yard(...",Weed Barrier Fabric,"Patio, Lawn & Garden",NaN,Agfabric,$87.3,NaN,NaN,Protect your yard and garden with our biodegra...,https://www.amazon.com/dp/B0CWH366KJ,https://m.media-amazon.com/images/I/71t3FD5KjH...,True,Only 5 left in stock - order soon.
1,B086L692VC,SAFAVIEH Braided Collection 4' Round Light Blu...,Area Rugs,Home & Kitchen,"50%jute, 25% Wool, 25% Cotton",Safavieh,$40.63,4.2,59.0,Country style is perfect for a casual cottage ...,https://www.amazon.com/dp/B086L692VC,https://m.media-amazon.com/images/I/A1Q73Cheh2...,True,Only 3 left in stock - order soon.
2,B01J6JELTG,Eyeseals 4.0 Sleep Mask – Clear – Moisturizing...,Sleeping Masks,Health & Household,Plastic,EYEECO,$65.95,3.7,1075.0,Locks moisture in: Eyeseals 4.0 eye mask for d...,https://www.amazon.com/dp/B01J6JELTG,https://m.media-amazon.com/images/I/61Uz393xlp...,True,NaN
3,B07HQSKK36,Lucky Monet 25/50/100PCS Burlap Gift Bags Wedd...,Gift Bags,Health & Household,Burlap,Lucky Monet,$29.99,4.6,2492.0,❤ Premium Burlap Material❤ These small burlap ...,https://www.amazon.com/dp/B07HQSKK36,https://m.media-amazon.com/images/I/71DrHIU1aM...,True,In Stock In Stock
4,B0C3Y8WJDR,St. Boniface Bag Company | Burlap Bags - Size:...,Grow Bags,"Patio, Lawn & Garden",5.0 Count,Generic,$29.99,4.4,11.0,100% Burlap > 100% BIODEGRADABLE AND ECO FRIEN...,https://www.amazon.com/dp/B0C3Y8WJDR,https://m.media-amazon.com/images/I/81q3el899U...,True,In Stock


# Preprocessing and Cleaning

In [5]:
# Check Null Values
df.isnull().sum()

id               0
title            0
name             0
category         0
material        14
brand            0
price            0
rating           6
reviewsCount     6
description      5
url              0
img_url          0
inStock          0
inStockText      4
dtype: int64

In [6]:
# Filling null values with empty string
for col in df.columns:
    df[col] = df[col].fillna('')

df.isnull().sum()

id              0
title           0
name            0
category        0
material        0
brand           0
price           0
rating          0
reviewsCount    0
description     0
url             0
img_url         0
inStock         0
inStockText     0
dtype: int64

In [7]:
# Cheack duplicate values 
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            100 non-null    object
 1   title         100 non-null    object
 2   name          100 non-null    object
 3   category      100 non-null    object
 4   material      100 non-null    object
 5   brand         100 non-null    object
 6   price         100 non-null    object
 7   rating        100 non-null    object
 8   reviewsCount  100 non-null    object
 9   description   100 non-null    object
 10  url           100 non-null    object
 11  img_url       100 non-null    object
 12  inStock       100 non-null    bool  
 13  inStockText   100 non-null    object
dtypes: bool(1), object(13)
memory usage: 10.4+ KB


In [9]:
df.head(2).T

,0,1
id,B0CWH366KJ,B086L692VC
title,"Agfabric Natural Jute Erosion Control, 16yard(...",SAFAVIEH Braided Collection 4' Round Light Blu...
name,Weed Barrier Fabric,Area Rugs
category,"Patio, Lawn & Garden",Home & Kitchen
material,,"50%jute, 25% Wool, 25% Cotton"
brand,Agfabric,Safavieh
price,$87.3,$40.63
rating,,4.2
reviewsCount,,59.0
description,Protect your yard and garden with our biodegra...,Country style is perfect for a casual cottage ...


In [10]:
# Defining function to clean text columns
def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^\w\s&\'%-]', '', text)
    text = re.sub(r'([!?.])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

In [11]:
text_columns = ["title", "name", "category", "material", "brand", "description"]

# Combine columns into a single text field
df["combined_text"] = df[[col for col in text_columns]].agg(" ".join, axis=1)

# Apply cleaning to combined_text
df["combined_text"] = df["combined_text"].apply(clean_text)
df.head(2).T

,0,1
id,B0CWH366KJ,B086L692VC
title,"Agfabric Natural Jute Erosion Control, 16yard(...",SAFAVIEH Braided Collection 4' Round Light Blu...
name,Weed Barrier Fabric,Area Rugs
category,"Patio, Lawn & Garden",Home & Kitchen
material,,"50%jute, 25% Wool, 25% Cotton"
brand,Agfabric,Safavieh
price,$87.3,$40.63
rating,,4.2
reviewsCount,,59.0
description,Protect your yard and garden with our biodegra...,Country style is perfect for a casual cottage ...


# Load a pretrained model

Here I have used **Sentence Transformer** **(BERT)** for **semantic search** in this e-commerce search engine project. By embedding user queries and product descriptions into semantic vectors, we achieve more accurate and context-aware search results. This approach effectively handles synonyms, variations, and multilingual inputs, significantly enhancing the user experience and driving better search outcomes.

In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model is running on: {model.device}")

Model is running on: cuda:0



# Generate Product Embeddings

In [13]:
product_embeddings = model.encode(df["combined_text"], convert_to_tensor = True)
product_embeddings[0]

tensor([-5.2350e-02,  4.2053e-02,  7.9948e-02, -2.3708e-02,  9.4810e-02,
         7.7652e-02,  3.4719e-02,  6.2199e-02, -2.5515e-02,  1.0498e-01,
        -2.4493e-02, -6.9093e-02, -3.1283e-02,  2.2911e-02, -2.3192e-02,
         9.6662e-02, -8.4870e-03,  5.5841e-02, -4.1746e-02, -1.5538e-02,
        -2.2079e-02,  6.7787e-02,  4.4334e-02, -6.1432e-03, -4.2369e-02,
        -3.8483e-03, -1.0020e-01,  6.9152e-02,  5.9988e-02, -8.7304e-02,
         3.2115e-02,  1.9188e-02, -1.9117e-02,  4.9270e-02, -1.8767e-02,
         2.0630e-02, -1.9463e-03, -5.6856e-02, -3.9037e-02,  3.2153e-02,
        -1.2050e-02, -4.3511e-03,  5.1067e-03, -6.5364e-02, -3.8571e-02,
         3.3141e-02,  1.1026e-02, -2.7701e-03,  8.3722e-02, -4.2876e-02,
         5.5093e-02,  1.3843e-02, -7.5928e-02,  5.9936e-04, -3.8881e-03,
        -4.2728e-02, -4.6735e-03,  3.9105e-02, -3.7215e-02,  2.9833e-02,
         4.7997e-02, -2.3406e-03, -8.8970e-02, -2.2958e-02, -3.2434e-02,
         3.2616e-02, -9.4565e-02,  7.1011e-02,  1.6

# Save Embeddings

In [14]:
# Save as .pkl
os.makedirs("output", exist_ok=True)
with open("output/sentence_transformer_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Model saved as sentence_transformer_model.pkl")

# Load it back for testing
with open("output/sentence_transformer_model.pkl", "rb") as f:
    model = pickle.load(f)
model = model.to(device) 
print(f"Loaded model is running on: {model.device}")

Model saved as sentence_transformer_model.pkl
Loaded model is running on: cuda:0


# Search Engine

In [15]:
from sentence_transformers.util import cos_sim

# Define a function for search product
def search_products(query, model, product_embeddings, df, top_k=5):
    # Ensure the dataset and embeddings are not empty
    if len(df) == 0 or len(product_embeddings) == 0:
        raise ValueError("Dataset or product embeddings are empty. Cannot perform search.")

    # Encode the query (returns a tensor on GPU)
    query_embedding = model.encode([clean_text(query)], convert_to_tensor=True)[0]

    # Convert query_embedding to NumPy array (move to CPU)
    query_embedding = query_embedding.cpu().numpy()

    # Ensure product_embeddings is a NumPy array
    if isinstance(product_embeddings, torch.Tensor):
        product_embeddings = product_embeddings.cpu().numpy()

    # Compute cosine similarity (both inputs are NumPy arrays)
    similarities = util.cos_sim(query_embedding, product_embeddings).flatten()

    # Debugging: Check the type and content of similarities
    print("Type of similarities:", type(similarities))
    print("Shape of similarities:", similarities.shape)
    print("Sample similarities:", similarities[:5])

    # Check for invalid similarities
    if len(similarities) == 0:
        raise ValueError("Cosine similarities are empty. Check dataset or query.")

    # Convert similarities to NumPy array if it’s not already
    if isinstance(similarities, torch.Tensor):
        similarities = similarities.cpu().numpy()

    # Check for NaN values
    if np.any(np.isnan(similarities)):
        raise ValueError("Cosine similarities contain NaN values. Check for zero embeddings in the dataset.")

    # Get top-k indices
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]

    # Return top-k products and scores
    return df.iloc[top_k_indices][["title", "brand", "description"]], similarities[top_k_indices]

# Test
query = "board games" # Straws, cutlary, toothbrush
try:
    top_products, scores = search_products(query, model, product_embeddings, df)

    # Create DataFrame 
    recommendations_df = pd.DataFrame(top_products)
    recommendations_df['Score'] = scores

    print("Query:", query)
    print("Top Recommendations:\n", recommendations_df)
except ValueError as e:
    print(f"Error: {e}")

Type of similarities: <class 'torch.Tensor'>
Shape of similarities: torch.Size([100])
Sample similarities: tensor([ 0.0394,  0.0514, -0.1040,  0.0411,  0.0658])
Query: board games
Top Recommendations:
                                                 title       brand  \
35  Partystadl 12 Pack Party Favor Boxes, Eco-Frie...  Partystadl   
50  Initial Canvas Tote Bag, Personalized Beach Ju...      shenee   
54  Reginary 16 Pcs Jute Tote Gift Bags Natural Bu...    Reginary   
91  Absorbent Drink Coasters Handmade Braided Drin...     XIAMOOR   
71  12pcs Halloween Heishi Beaded Bracelets for Wo...      YWOYWO   

                                          description     Score  
35  Customizable and Versatile – These DIY treat b...  0.241385  
50                                                     0.221722  
54                                                     0.209816  
91  🍷 BETTER PROTECTS YOUR FURNITURE- Thicker(8 mm...  0.201134  
71                                                   